# Healthier Recipe Optimization via Deep Neural Networks & Metric Loss
**Author:** Kristian Hudák  
**Competition:** National Olympiad in Artificial Intelligence (OAI) — Home Qualifier Round

This notebook explores an autonomous Machine Learning approach to culinary optimization:
1. **Nutritional Vector Embedding:** Transforming raw ingredients into continuous 8-dimensional nutrient vectors via fuzzy matching against the USDA FoodData Central database.
2. **Neural Multi-Attribute Mapping (`AlternativnaSiet`):** A PyTorch multi-layer perceptron trained to map unhealthy ingredient profiles directly into idealized healthy target vectors.
3. **Multi-Objective Penalty Function:** A customized non-linear cost function that heavily penalizes excess calories, carbohydrates, and fats while rewarding preserved protein content.
4. **Iterative Greedy Recipe Optimization:** Greedily selecting and substituting ingredients to reach targeted caloric deficit targets while preserving nutritional balance.


In [2]:
# 1. Environment setup, library imports, and dataset ingestion
!pip install thefuzz

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils import data as data_utils
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial import distance
from thefuzz import process, fuzz
import copy

# Load paired healthy/unhealthy ingredient dataset
df_pairs = pd.read_csv('/content/priklad na zdravsie suroviny.csv')
df_pairs = df_pairs.rename(columns={
    'Unhealthy Input (High Penalty)': 'Unhealthy Input',
    'Healthy Target (Health Direction)': 'Healthy Target'
})

df_ingr = pd.read_excel('/content/pivoted_categorized_food_data.xlsx')

print("✅ Dáta načítané")
print(f"📊 Páry receptov: {len(df_pairs)}")
print(f"📊 Ingrediencie v databáze: {len(df_ingr)}")
df_pairs.head()


✅ Dáta načítané
📊 Páry receptov: 24
📊 Ingrediencie v databáze: 321


,Unhealthy Input,Healthy Target,Category ID
0,Mayonnaise,Non-fat Greek Yogurt,4
1,Tartar sauce,Herb Yogurt Sauce,6
2,Heavy cream 33%,Sour cream 10%,1
3,Butter,Mashed Avocado,4
4,Pork lard,Olive oil,4


In [3]:
# 2. Vectorize text names to nutrient profiles via fuzzy string matching
def ziskaj_vektor(nazov, df_ref):
    """Retrieves 8-dimensional nutrient vector for ingredient via token fuzzy matching"""
    match, score, idx = process.extractOne(nazov, df_ref['description'], scorer=fuzz.token_set_ratio)

    if score > 60:
        return df_ref.iloc[idx, 1:9].values.astype(float)
    else:
        return None

def spracuj_par(row, df_ref):
    """Processes dataset row and extracts nutrient feature vectors"""
    nazov_nezdrave = row['Unhealthy Input']
    nazov_zdrave = row['Healthy Target']

    vec_in = ziskaj_vektor(nazov_nezdrave, df_ref)
    vec_out = ziskaj_vektor(nazov_zdrave, df_ref)

    if vec_in is not None and vec_out is not None:
        if not (isinstance(vec_in, np.ndarray) and vec_in.shape == (8,)) or \
           not (isinstance(vec_out, np.ndarray) and vec_out.shape == (8,)):
            return pd.Series([None, None])
        return pd.Series([vec_in, vec_out])
    else:
        return pd.Series([None, None])

print("⏳ Začínam párovanie ingrediencií...")

df_pairs[['vector_in', 'vector_out']] = df_pairs.apply(
    lambda row: spracuj_par(row, df_ingr), axis=1
)

df_clean = df_pairs.dropna(subset=['vector_in', 'vector_out'])

if len(df_clean) > 0:
    X_numpy = np.stack(df_clean['vector_in'].values)
    y_numpy = np.stack(df_clean['vector_out'].values)

    X_train = torch.tensor(X_numpy, dtype=torch.float32)
    y_train = torch.tensor(y_numpy, dtype=torch.float32)

print(f"✅ Vytvorených {len(df_clean)} trénovacích párov")


⏳ Začínam párovanie ingrediencií...
✅ Vytvorených 12 trénovacích párov


In [4]:
# 3. Train/validation/test splitting (70/15/15) and MinMaxScaler normalization
X_full_np = X_train.numpy()
y_full_np = y_train.numpy()

# Split 70% train, 15% val, 15% test
X_train_np, X_temp_np, y_train_np, y_temp_np = train_test_split(
    X_full_np, y_full_np, test_size=0.30, random_state=42
)

X_val_np, X_test_np, y_val_np, y_test_np = train_test_split(
    X_temp_np, y_temp_np, test_size=0.5, random_state=42
)

# Feature normalization
scaler_X = MinMaxScaler()
X_train_scaled_np = scaler_X.fit_transform(X_train_np)
X_val_scaled_np = scaler_X.transform(X_val_np)
X_test_scaled_np = scaler_X.transform(X_test_np)

scaler_y = MinMaxScaler()
y_train_scaled_np = scaler_y.fit_transform(y_train_np.reshape(-1, y_train_np.shape[-1]))
y_val_scaled_np = scaler_y.transform(y_val_np.reshape(-1, y_val_np.shape[-1]))
y_test_scaled_np = scaler_y.transform(y_test_np.reshape(-1, y_test_np.shape[-1]))

# Konverzia na tensory
X_train = torch.from_numpy(X_train_scaled_np).float()
y_train = torch.from_numpy(y_train_scaled_np).float()
X_val = torch.from_numpy(X_val_scaled_np).float()
y_val = torch.from_numpy(y_val_scaled_np).float()
X_test = torch.from_numpy(X_test_scaled_np).float()
y_test = torch.from_numpy(y_test_scaled_np).float()

# DataLoadery
batch_size = 32

train_tensor = data_utils.TensorDataset(X_train, y_train)
train_dataloader = data_utils.DataLoader(dataset=train_tensor, batch_size=batch_size, shuffle=True)

valid_tensor = data_utils.TensorDataset(X_val, y_val)
valid_dataloader = data_utils.DataLoader(dataset=valid_tensor, batch_size=1, shuffle=False)

test_tensor = data_utils.TensorDataset(X_test, y_test)
test_dataloader = data_utils.DataLoader(dataset=test_tensor, batch_size=1, shuffle=False)

print(f"✅ Train set: X={X_train.shape}, y={y_train.shape}")
print(f"✅ Validation set: X={X_val.shape}, y={y_val.shape}")
print(f"✅ Test set: X={X_test.shape}, y={y_test.shape}")


✅ Train set: X=torch.Size([8, 8]), y=torch.Size([8, 8])
✅ Validation set: X=torch.Size([2, 8]), y=torch.Size([2, 8])
✅ Test set: X=torch.Size([2, 8]), y=torch.Size([2, 8])


In [5]:
# 4. PyTorch neural network architecture and MSE loss training loop
class AlternativnaSiet(nn.Module):
    def __init__(self):
        super(AlternativnaSiet, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(8, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 8)
        )

    def forward(self, x):
        return self.fc(x)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Inference device: {device}")

model = AlternativnaSiet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

epochs = 200

print(f"🚀 Začínam trénovanie na {epochs} epoch...")

loss_stats_MSE = {'train': [], "val": []}
loss_stats_MAE = {'train': [], "val": []}

best_model = None
best_valid_MSE = float('inf')

for epoch in range(epochs):
    model.train()
    running_loss_mse_train = 0.0
    running_mae_train = 0.0

    for i, data in enumerate(train_dataloader):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        y = model(inputs)

        MAE = abs(labels-y).mean()
        running_mae_train += MAE.item()

        loss = criterion(y, labels)
        loss.backward()
        optimizer.step()
        running_loss_mse_train += loss.item()

    running_loss_mse_val = 0.0
    running_mae_val = 0.0

    with torch.no_grad():
        model.eval()
        for X_val_batch, y_val_batch in valid_dataloader:
            X_val_batch, y_val_batch = X_val_batch.to(device), y_val_batch.to(device)

            y_val_pred = model(X_val_batch)
            val_loss = criterion(y_val_pred, y_val_batch)
            running_loss_mse_val += val_loss.item()

            MAE = abs(y_val_pred-y_val_batch).mean()
            running_mae_val += MAE.item()

    MSE_train = running_loss_mse_train/len(train_dataloader)
    loss_stats_MSE['train'].append(MSE_train)
    MSE_val = running_loss_mse_val/len(valid_dataloader)
    loss_stats_MSE['val'].append(MSE_val)
    MAE_train = running_mae_train/len(train_dataloader)
    loss_stats_MAE['train'].append(MAE_train)
    MAE_val = running_mae_val/len(valid_dataloader)
    loss_stats_MAE['val'].append(MAE_val)

    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1}: TRAIN MSE={MSE_train:0.2f} MAE={MAE_train:0.2f} | VALID MSE={MSE_val:0.2f} MAE={MAE_val:0.2f}')

    if MSE_val <= best_valid_MSE:
        best_valid_MSE = MSE_val
        best_model = copy.deepcopy(model)

print("✅ Trénovanie dokončené!")


🖥️  Používam zariadenie: cpu
🚀 Začínam trénovanie na 200 epoch...
Epoch 20: TRAIN MSE=0.22 MAE=0.34 | VALID MSE=0.24 MAE=0.36
Epoch 40: TRAIN MSE=0.10 MAE=0.25 | VALID MSE=0.22 MAE=0.38
Epoch 60: TRAIN MSE=0.07 MAE=0.21 | VALID MSE=0.23 MAE=0.36
Epoch 80: TRAIN MSE=0.05 MAE=0.18 | VALID MSE=0.32 MAE=0.39
Epoch 100: TRAIN MSE=0.04 MAE=0.16 | VALID MSE=0.41 MAE=0.44
Epoch 120: TRAIN MSE=0.03 MAE=0.14 | VALID MSE=0.51 MAE=0.49
Epoch 140: TRAIN MSE=0.03 MAE=0.12 | VALID MSE=0.63 MAE=0.54
Epoch 160: TRAIN MSE=0.02 MAE=0.11 | VALID MSE=0.72 MAE=0.57
Epoch 180: TRAIN MSE=0.02 MAE=0.10 | VALID MSE=0.77 MAE=0.59
Epoch 200: TRAIN MSE=0.01 MAE=0.08 | VALID MSE=0.79 MAE=0.59
✅ Trénovanie dokončené!


In [6]:
# 5. Multi-objective penalty loss function for ingredient candidate selection
def vypocitaj_penalty_v2(vektor_suroviny, target_kcal, target_prot, target_carb,
                         current_kcal, current_prot, current_carb):
    kcal, protein, carbs, fat = (
        vektor_suroviny[0],
        vektor_suroviny[1],
        vektor_suroviny[2],
        vektor_suroviny[3]
    )
    base_penalty = (kcal * 0.5) + (carbs * 1.2) + (fat * 0.1) - (protein * 1.2)
    kcal_gap = max(0, current_kcal - target_kcal) / target_kcal if target_kcal > 0 else 0
    prot_gap = max(0, target_prot - current_prot) / target_prot if target_prot > 0 else 0
    carb_gap = max(0, current_carb - target_carb) / target_carb if target_carb > 0 else 0

    gap_penalty = 0
    if current_kcal > target_kcal:
        gap_penalty += kcal * kcal_gap * 2.0
    if current_carb > target_carb:
        gap_penalty += carbs * carb_gap * 1.5
    if current_prot < target_prot:
        gap_penalty -= protein * prot_gap * 1.0

    return base_penalty + gap_penalty

def optimalizuj_surovinu(row_idx, recept_df, model, df_ingr, scaler_X, scaler_y, device, debug=False):
    worst_vec = recept_df.loc[row_idx].iloc[1:9].values.astype(float)
    v_in_scaled = scaler_X.transform(worst_vec.reshape(1, -1))

    model.eval()
    with torch.no_grad():
        input_tensor = torch.from_numpy(v_in_scaled).float().to(device)
        v_out_scaled = model(input_tensor).cpu().numpy()

    v_target = scaler_y.inverse_transform(v_out_scaled)
    cat_id = recept_df.loc[row_idx, 'food_category_id']
    potential_candidates = df_ingr[df_ingr['food_category_id'] == cat_id]

    if len(potential_candidates) < 5:
        potential_candidates = df_ingr

    candidate_vectors = potential_candidates.iloc[:, 1:9].values
    distances = distance.cdist(v_target, candidate_vectors, 'euclidean')

    best_idx = distances.argmin()
    best_candidate = potential_candidates.iloc[best_idx]

    if debug:
        print(f"  🔄 Náhrada: {recept_df.loc[row_idx, 'description']} -> {best_candidate['description']}")

    return best_candidate

def najdi_nahodnu_nahradu(row_idx, recept_df, df_ingr, target_low_cal=True, target_high_prot=True):
    cat_id = recept_df.loc[row_idx, 'food_category_id']
    candidates = df_ingr[df_ingr['food_category_id'] == cat_id]

    if len(candidates) < 3:
        candidates = df_ingr
    if target_low_cal:
        candidates = candidates[candidates['calories'] < recept_df.loc[row_idx, 'calories']]
    if target_high_prot:
        candidates = candidates[candidates['protein'] > recept_df.loc[row_idx, 'protein'] * 0.8]

    if candidates.empty:
        candidates = df_ingr[df_ingr['food_category_id'] == cat_id]
        if candidates.empty:
            candidates = df_ingr

    return candidates.sample(1).iloc[0]

print("✅ Pomocné funkcie načítané")


✅ Pomocné funkcie načítané


In [7]:
# 6. Iterative recipe-wide optimization loop (caloric reduction & protein retention)
def optimalizuj_recept(recept_df, model, df_ingr, scaler_X, scaler_y, device,
                       max_iterations=50, verbose=True):
    povodny_recept = recept_df.copy()
    aktualny_recept = recept_df.copy()
    nahradene_indexy = set()

    povodna_energia = povodny_recept['calories'].sum()
    povodny_protein = povodny_recept['protein'].sum()
    povodny_carb = povodny_recept['carbs'].sum()

    target_kcal = povodna_energia * 0.75
    target_prot = povodny_protein
    target_carb = povodny_carb

    if verbose:
        print(f"\n🎯 CIELE: Kalórie ≤ {target_kcal:.1f} | Proteín ≥ {target_prot:.1f}g | Sacharidy < {target_carb:.1f}g")

    stuck_counter = 0
    last_kcal = povodna_energia

    for iteration in range(max_iterations):
        akt_kcal = aktualny_recept['calories'].sum()
        akt_prot = aktualny_recept['protein'].sum()
        akt_carb = aktualny_recept['carbs'].sum()

        if (akt_kcal <= target_kcal and akt_prot >= target_prot and akt_carb < target_carb):
            if verbose: print(f"✅ ÚSPECH v iterácii {iteration+1}!")
            break

        if abs(akt_kcal - last_kcal) < 5: stuck_counter += 1
        else: stuck_counter = 0
        last_kcal = akt_kcal

        use_fallback = stuck_counter > 5 or iteration > 30

        temp_recept = aktualny_recept.copy()
        temp_recept['penalty'] = temp_recept.apply(
            lambda row: vypocitaj_penalty_v2(row.iloc[1:9].values, target_kcal, target_prot, target_carb, akt_kcal, akt_prot, akt_carb), axis=1
        )

        moznosti = temp_recept[~temp_recept.index.isin(nahradene_indexy)]
        if moznosti.empty:
            nahradene_indexy.clear()
            moznosti = temp_recept

        idx_na_nahradu = moznosti['penalty'].idxmax()

        if use_fallback:
            nova_surovina = najdi_nahodnu_nahradu(idx_na_nahradu, aktualny_recept, df_ingr, (akt_kcal > target_kcal), (akt_prot < target_prot))
        else:
            nova_surovina = optimalizuj_surovinu(idx_na_nahradu, aktualny_recept, model, df_ingr, scaler_X, scaler_y, device, (verbose and iteration % 10 == 0))

        for col in aktualny_recept.columns:
            if col in nova_surovina.index:
                aktualny_recept.at[idx_na_nahradu, col] = nova_surovina[col]

        nahradene_indexy.add(idx_na_nahradu)

    return povodny_recept, aktualny_recept, nahradene_indexy

print("✅ Optimalizačná funkcia pripravená")


✅ Optimalizačná funkcia pripravená


In [9]:
# 7. Interactive console interface for recipe testing
def rucny_vstup_receptu(df_ingr):
    print("\n--- INTERACTIVE RECIPE INPUT ---")
    vstup = input("Enter ingredients separated by commas: ")
    if vstup.lower() == 'späť' or not vstup: return None

    polozky = [p.strip() for p in vstup.split(',') if p.strip()]
    vybrate_suroviny = []

    for p in polozky:
        match, score, idx = process.extractOne(p, df_ingr['description'], scorer=fuzz.token_set_ratio)
        if score > 55:
            print(f"   ✅ Identified: '{p}' -> '{match}'")
            vybrate_suroviny.append(df_ingr.iloc[idx])

    return pd.DataFrame(vybrate_suroviny).reset_index(drop=True) if vybrate_suroviny else None

def main():
    print("=" * 60 + "\n Salad AI NUTRIČNÝ OPTIMALIZÁTOR V2.0\n" + "=" * 60)
    while True:
        volba = input("\nStlač ENTER pre recept alebo napíš 'koniec': ").lower()
        if volba == 'koniec': break

        recept_df = rucny_vstup_receptu(df_ingr)
        if recept_df is None: continue

        povodny, optimalizovany, zmeny = optimalizuj_recept(recept_df, best_model, df_ingr, scaler_X, scaler_y, device)

        print("\n📊 POROVNANIE HODNÔT")
        stats = []
        for n in ['calories', 'protein', 'carbs', 'fat']:
            p_val, a_val = povodny[n].sum(), optimalizovany[n].sum()
            zmena = ((a_val - p_val) / p_val * 100) if p_val else 0
            stats.append({'Nutrient': n.capitalize(), 'Pôvodné': f"{p_val:.2f}", 'Alternatíva': f"{a_val:.2f}", 'Zmena (%)': f"{zmena:+.1f}%"})

        print(pd.DataFrame(stats).to_string(index=False))
        print("\n🔄 Zoznam zmien:")
        for idx in zmeny:
            if povodny.loc[idx, 'description'] != optimalizovany.loc[idx, 'description']:
                print(f"  • {povodny.loc[idx, 'description']} ➡️  {optimalizovany.loc[idx, 'description']}")

if __name__ == "__main__":
    main()


 Salad AI NUTRIČNÝ OPTIMALIZÁTOR V2.0

Stlač ENTER pre recept alebo napíš 'koniec': Pork loin raw, Sweet potatoes raw, Green beans raw, Sour cream

--- RUČNÉ ZADÁVANIE RECEPTU ---
Zadaj suroviny oddelené čiarkou: Pork loin raw, Sweet potatoes raw, Green beans raw, Sour cream
   ✅ Rozpoznané: 'Pork loin raw' -> 'Pork, loin, boneless, raw'
   ✅ Rozpoznané: 'Sweet potatoes raw' -> 'Sweet potatoes, orange flesh, without skin, raw'
   ✅ Rozpoznané: 'Green beans raw' -> 'Beans, snap, green, raw'
   ✅ Rozpoznané: 'Sour cream' -> 'Cream, sour, full fat'

🎯 CIELE: Kalórie ≤ 362.4 | Proteín ≥ 27.7g | Sacharidy < 30.3g
  🔄 Náhrada: Cream, sour, full fat -> Cheese, ricotta, whole milk
  🔄 Náhrada: Ham, sliced, restaurant -> Ham, sliced, restaurant
  🔄 Náhrada: Corn, sweet, yellow and white kernels,  fresh, raw -> Corn, sweet, yellow and white kernels,  fresh, raw
  🔄 Náhrada: Corn, sweet, yellow and white kernels,  fresh, raw -> Corn, sweet, yellow and white kernels,  fresh, raw
✅ ÚSPECH v iteráci

KeyboardInterrupt: Interrupted by user

### Benchmark Recipes for Interactive Testing
- `Pork loin raw, Sweet potatoes raw, Green beans raw, Sour cream`
- `Oatmeal raw, Milk whole, Banana raw, Peanut butter`
- `Pork loin, Potatoes, Cream, Lard, Onion`
